# OpenNyAI rhetorical-role tagging

This notebook runs OpenNyAI's **pretrained** rhetorical-role tagger over every non-empty judgment in `IN-Abs-prepared/train/judgement`. 

It writes resumable JSONL checkpoints to `opennyai_tagged_all_train/`.


### Setup
Imports the required libraries and confirm CPU or GPU Usage

In [8]:
import csv
import json
import sys
import time
from pathlib import Path

import torch
import spacy
from opennyai import Pipeline
from opennyai.utils import Data

if sys.version_info < (3, 13):
    raise RuntimeError(
        f"OpenNyAI 0.0.15 requires Python 3.13+. "
        f"Current: {sys.version.split()[0]}"
    )


spacy.require_cpu()
USE_GPU = False

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("OpenNyAI device: CPU")

Python: 3.13.17
PyTorch: 2.11.0+cu128
CUDA available: True
OpenNyAI device: CPU


### Load judgments
Finds the judgment files, prepares the output folder, and selects the documents to tag.

In [9]:
DATA_ROOT = Path("IN-Abs-prepared")
if not DATA_ROOT.exists() and (Path.cwd() / "IN-Abs-prepared").exists():
    DATA_ROOT = Path.cwd() / "IN-Abs-prepared"

judgment_dir = DATA_ROOT / "train" / "judgement"
if not judgment_dir.is_dir():
    judgment_dir = DATA_ROOT / "train" / "judgment"

if not judgment_dir.is_dir():
    raise FileNotFoundError(
        f"Could not find train/judgement under {DATA_ROOT.resolve()}. "
        "Put this notebook beside IN-Abs-prepared."
    )

OUTPUT_DIR = Path("opennyai_trial_cpu_200")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = OUTPUT_DIR / "opennyai_cpu_200.jsonl"
CSV_PATH = OUTPUT_DIR / "opennyai_cpu_200_sentence_tags.csv"

judgment_files = sorted(
    judgment_dir.glob("*.txt"),
    key=lambda p: (0, int(p.stem)) if p.stem.isdigit() else (1, p.stem.lower()),
)

records = []
for path in judgment_files:
    text = path.read_text(encoding="utf-8", errors="replace").strip()
    if text:
        records.append({
            "doc_id": path.stem,
            "source_file": str(path),
            "judgment": text,
        })

print("Judgment files found:", len(judgment_files))
print("Non-empty judgments found:", len(records))

if len(records) != 7030:
    print(
        f"NOTE: expected 7,030 non-empty judgments, found {len(records)}. "
        "Confirm DATA_ROOT points to the prepared IN-Abs dataset."
    )

if not records:
    raise ValueError("No non-empty judgment files found.")

print("Judgments to tag:", len(records))
print("First IDs:", [record["doc_id"] for record in records[:5]])
print("Trial output folder:", OUTPUT_DIR.resolve())


Judgment files found: 7030
Non-empty judgments found: 7030
Judgments to tag: 7030
First IDs: ['1', '2', '3', '4', '5']
Trial output folder: C:\Users\pdipu\Downloads\legallens\opennyai_trial_cpu_200


### Load pretrained tagger
Loads OpenNyAI’s pretrained rhetorical-role model for tagging.

In [10]:
role_pipeline = Pipeline(components=["Rhetorical_Role"], use_gpu=USE_GPU, verbose=True)
print("OpenNyAI rhetorical-role tagger is ready.")


ℹ Loading Rhetorical Role...
ℹ Rhetorical Roles will use CPU!


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 250.44it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OpenNyAI rhetorical-role tagger is ready.


In [11]:
print("role_pipeline" in globals())

True


### Tag and checkpoint results
Tag each judgment, save its sentence roles, and record progress so completed documents can be skipped if the cell is rerun.

In [12]:
TRIAL_DIR = Path("opennyai_trial_cpu_200")
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_JSONL_PATH = TRIAL_DIR / "tagged.jsonl"
TRIAL_CSV_PATH = TRIAL_DIR / "sentence_roles.csv"

trial_records = records[:1200]
print("Judgments in this trial:", len(trial_records))


def make_json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if hasattr(value, "to_dict"):
        return make_json_safe(value.to_dict())
    return str(value)


def extract_sentence_roles(prediction):
    output = []
    annotations = (
        prediction.get("annotations", [])
        if isinstance(prediction, dict)
        else []
    )

    for annotation in annotations:
        if not isinstance(annotation, dict):
            continue

        sentence = annotation.get("text") or annotation.get("sentence")
        labels = annotation.get("labels")
        if not sentence or labels is None:
            continue

        if isinstance(labels, (list, tuple)):
            role = ", ".join(
                str(label) for label in labels if label is not None
            )
        else:
            role = str(labels)

        if role:
            output.append({"sentence": sentence, "role": role})

    return output


completed_ids = set()

if TRIAL_JSONL_PATH.exists():
    with TRIAL_JSONL_PATH.open(encoding="utf-8") as input_file:
        for line in input_file:
            try:
                saved = json.loads(line)
                if saved.get("doc_id") is not None:
                    completed_ids.add(str(saved["doc_id"]))
            except json.JSONDecodeError:
                continue

todo = [
    record
    for record in trial_records
    if str(record["doc_id"]) not in completed_ids
]

print(f"Already completed: {len(completed_ids)} | Remaining: {len(todo)}")

started = time.perf_counter()

with TRIAL_JSONL_PATH.open("a", encoding="utf-8") as output_file:
    for offset, record in enumerate(todo, start=1):
        result = role_pipeline(
            Data([record["judgment"]], use_gpu=False)
        )

        prediction = (
            result[0]
            if isinstance(result, (list, tuple)) and result
            else result
        )
        prediction = make_json_safe(prediction)
        sentence_roles = extract_sentence_roles(prediction)

        tagged_judgment = "\n".join(
            f"[{item['role']}] {item['sentence']}"
            for item in sentence_roles
        )

        row = {
            "doc_id": str(record["doc_id"]),
            "source_file": record.get("source_file"),
            "prediction": prediction,
            "sentence_roles": sentence_roles,
            "tagged_judgment": tagged_judgment,
        }

        output_file.write(json.dumps(row, ensure_ascii=False) + "\n")
        output_file.flush()
        completed_ids.add(str(record["doc_id"]))

        if offset % 25 == 0 or offset == len(todo):
            elapsed = time.perf_counter() - started
            average = elapsed / offset
            remaining_seconds = average * (len(todo) - offset)

            print(
                f"{len(completed_ids)}/{len(trial_records)} completed | "
                f"latest judgment: {record['doc_id']} "
                f"({len(sentence_roles)} sentence tags) | "
                f"elapsed: {elapsed / 3600:.2f} h | "
                f"estimated remaining: {remaining_seconds / 3600:.2f} h"
            )


with TRIAL_CSV_PATH.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=["doc_id", "sentence", "role"],
    )
    writer.writeheader()

    with TRIAL_JSONL_PATH.open(encoding="utf-8") as input_file:
        for line in input_file:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            for item in row.get("sentence_roles", []):
                writer.writerow({
                    "doc_id": row.get("doc_id"),
                    "sentence": item.get("sentence"),
                    "role": item.get("role"),
                })

print(f"Finished: {len(completed_ids)} of {len(trial_records)} judgments")
print("JSONL checkpoint:", TRIAL_JSONL_PATH.resolve())
print("Sentence-role CSV:", TRIAL_CSV_PATH.resolve())

Judgments in this trial: 1200
Already completed: 600 | Remaining: 600
ℹ Pre-processing will happen on CPU!
ℹ Preprocessing rhetorical role model input!!!


100%|██████████| 1/1 [00:24<00:00, 24.86s/it]


ℹ Processing documents with rhetorical role model!!!


100%|██████████| 1/1 [00:35<00:00, 35.40s/it]


ℹ Pre-processing will happen on CPU!
ℹ Preprocessing rhetorical role model input!!!


100%|██████████| 1/1 [00:38<00:00, 38.23s/it]


... (2,391 outputs left out)

100%|██████████| 1/1 [00:10<00:00, 10.91s/it]


ℹ Processing documents with rhetorical role model!!!


100%|██████████| 1/1 [00:19<00:00, 19.46s/it]


1200/1200 completed | latest judgment: 1215 (73 sentence tags) | elapsed: 10.41 h | estimated remaining: 0.00 h
Finished: 1200 of 1200 judgments
JSONL checkpoint: C:\Users\pdipu\Downloads\legallens\opennyai_trial_cpu_200\tagged.jsonl
Sentence-role CSV: C:\Users\pdipu\Downloads\legallens\opennyai_trial_cpu_200\sentence_roles.csv


### Check tagging results
Counts the saved judgments and sentence-role tags to confirm that output was written.

In [7]:
with TRIAL_JSONL_PATH.open(encoding="utf-8") as f:
    saved_rows = [json.loads(line) for line in f if line.strip()]

print("Judgments saved:", len(saved_rows))
print(
    "Sentence-role tags saved:",
    sum(len(row.get("sentence_roles", [])) for row in saved_rows),
)
print("Example judgment ID:", saved_rows[0]["doc_id"] if saved_rows else "none")
print(
    "Example tags:",
    saved_rows[0].get("sentence_roles", [])[:3] if saved_rows else [],
)


Judgments saved: 600
Sentence-role tags saved: 85792
Example judgment ID: 1
Example tags: [{'sentence': 'Appeal No. Lxvi of 1949.', 'role': 'PREAMBLE'}, {'sentence': '\nAppeal from the High Court of judicature, Bombay, in a reference under section 66 of the Indian Income tax Act, 1022.\nK.M. Munshi (N. P. Nathvani, with him), for the appel lant.', 'role': 'PREAMBLE'}, {'sentence': "' M.C. Setalvad, Attorney General for India (H. J. Umrigar, with him), for the respondent.", 'role': 'NONE'}]


In [14]:
import json
from collections import Counter
from pathlib import Path

RESULT_PATH = Path(
    globals().get(
        "TRIAL_JSONL_PATH",
        "opennyai_trial_cpu_200/tagged.jsonl",
    )
)

if not RESULT_PATH.is_file():
    raise FileNotFoundError(f"Tagged JSONL file not found: {RESULT_PATH.resolve()}")

rows = []
with RESULT_PATH.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue

role_counts = Counter()
docs_with_role = Counter()

for row in rows:
    seen_roles_in_doc = set()

    for item in row.get("sentence_roles", []):
        raw_role = item.get("role", "")
        roles = [role.strip() for role in str(raw_role).split(",") if role.strip()]

        for role in roles:
            role_counts[role] += 1
            seen_roles_in_doc.add(role)

    for role in seen_roles_in_doc:
        docs_with_role[role] += 1

num_docs = len({str(row.get("doc_id", "unknown")) for row in rows})
num_sentences = sum(len(row.get("sentence_roles", [])) for row in rows)
total_assignments = sum(role_counts.values())

print("Tagged judgments:", num_docs)
print("Tagged sentences:", num_sentences)
print("Role assignments:", total_assignments)
print("\nRole distribution:")

for role, count in role_counts.most_common():
    percent = 100 * count / total_assignments if total_assignments else 0
    doc_percent = 100 * docs_with_role[role] / num_docs if num_docs else 0
    print(
        f"{role:20s} {count:7d} tags ({percent:5.1f}%) | "
        f"present in {docs_with_role[role]:4d}/{num_docs} judgments "
        f"({doc_percent:5.1f}%)"
    )

Tagged judgments: 1200
Tagged sentences: 161516
Role assignments: 161516

Role distribution:
ANALYSIS               91297 tags ( 56.5%) | present in 1199/1200 judgments ( 99.9%)
FAC                    25344 tags ( 15.7%) | present in 1184/1200 judgments ( 98.7%)
PRE_RELIED             16038 tags (  9.9%) | present in  919/1200 judgments ( 76.6%)
STA                     6536 tags (  4.0%) | present in  884/1200 judgments ( 73.7%)
NONE                    4111 tags (  2.5%) | present in 1181/1200 judgments ( 98.4%)
RLC                     3710 tags (  2.3%) | present in  940/1200 judgments ( 78.3%)
RPC                     3411 tags (  2.1%) | present in 1165/1200 judgments ( 97.1%)
RATIO                   3266 tags (  2.0%) | present in 1142/1200 judgments ( 95.2%)
ARG_PETITIONER          2815 tags (  1.7%) | present in  668/1200 judgments ( 55.7%)
PREAMBLE                2574 tags (  1.6%) | present in 1176/1200 judgments ( 98.0%)
ARG_RESPONDENT          1178 tags (  0.7%) | present in  